Evaluates **baseline (full tokens) vs document-side compression** at ratios 10%–90%
for four strategies: `random`, `pool1d`, `pool2d`, `hier`.
 
Compression is applied **on-the-fly** during evaluation (no intermediate pkl files).
Pre-encoded ViDoRe PKL files (from Pipeline A) are loaded directly.
Results are saved to Excel.

In [1]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt


Looking in links: /kaggle/input/datasets/nhhsag12/colpali-dependency
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/colpali_engine-0.3.15-py3-none-any.whl (from -r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 1))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/colbert_ai-0.2.21-py3-none-any.whl (from -r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/bitarray-3.8.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (from colbert-ai==0.2.21->-r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/git_python-1.0.3-py2.py3-none-any.whl (from colbert-ai==0.2.21->-r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/transformers-5.4.0-py3-no

## Cell 1 — Imports & Config

In [2]:
# ---- Paths --------------------------------------------------------------
# Pre-encoded ViDoRe PKL files produced by Pipeline A (one pkl per domain)
PAGE_PKL_DIR  = "/kaggle/input/datasets/guiniever/vidore-encoded"

# ViDoRe V3 root — used to discover corpus / queries / qrels parquets per domain.
# Layout: {VIDORE_ROOT}/vidore_v3_{domain}/vidore_v3_{domain}/corpus/*.parquet
# Corpus parquets contain: corpus_id, image (HF image dict), doc_id, page_number_in_doc
VIDORE_ROOT   = "/kaggle/input/datasets/namthi/vidore-v3"

COLQWEN2_BASE = "/kaggle/input/models/nhhsag12/colqwen2-v1-0-base/pytorch/default/1"
COLQWEN2_LORA = "/kaggle/input/models/nhhsag12/colqwen2-v1-0/pytorch/default/2"

WORKING_DIR   = "/kaggle/working"

# ---- Compression --------------------------------------------------------
# Fraction of tokens KEPT (0.1 = keep 10%, drop 90%)
KEEP_RATIOS  = [round(i * 0.1, 1) for i in range(1, 10)]   # 0.1 … 0.9
STRATEGIES   = ["random", "pool1d", "pool2d", "hier"]

# pool2d / hier need exact per-page patch grids.
# Grids are computed from raw images loaded from the corpus parquets (same
# images that were encoded during Pipeline A), via smart_resize_local().
# Set False only if pool2d and hier are removed from STRATEGIES.
NEED_GRID = ("pool2d" in STRATEGIES) or ("hier" in STRATEGIES)

# Model-specific patch geometry (ColQwen2)
PATCH_SIZE  = 14
MERGE_SIZE  = 2

# Hierarchical Ward settings
HIER_DEVICE     = "cuda"   # or "cpu"
HIER_BATCH_SIZE = 1024     # pages per GPU batch

# Random pruning
SEED = 0

# ---- Evaluation ---------------------------------------------------------
TOPK_EVAL = [1, 3, 5, 10]

# smart_resize constants (mirrors ColQwen2Retriever)
FACTOR     = 28
MIN_PIXELS = 4  * 28 * 28
MAX_PIXELS = 768 * 28 * 28
MAX_RATIO  = 200

## Cell 2 — Imports


In [3]:
import gc
import glob
import io
import json
import math
import os
import pickle
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Device: cuda


## Cell 3 — ColQwen2 Model Definitions


In [4]:
# ==============================================================================
# ColQwen2 & ColQwen2Processor — defined from scratch (no colpali_engine import)
# Sources provided by user.
# ==============================================================================

import importlib
from abc import ABC, abstractmethod
from typing import ClassVar, List, Optional, Tuple, Union

import torch
from torch import nn
from PIL import Image
from transformers import BatchEncoding, BatchFeature
from transformers.models.qwen2_vl import Qwen2VLConfig, Qwen2VLModel, Qwen2VLProcessor
from transformers.models.qwen2_vl.image_processing_qwen2_vl import smart_resize


# ── Minimal device helper (replaces colpali_engine.utils.get_torch_device) ────
def _get_torch_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device)


# ==============================================================================
# BaseVisualRetrieverProcessor
# ==============================================================================

class BaseVisualRetrieverProcessor(ABC):
    """Base class for visual retriever processors."""

    query_prefix: ClassVar[str] = ""

    @abstractmethod
    def process_images(self, images: List[Image.Image]) -> Union[BatchFeature, BatchEncoding]:
        pass

    @abstractmethod
    def process_texts(self, texts: List[str]) -> Union[BatchFeature, BatchEncoding]:
        pass

    def process_queries(
        self,
        texts: Optional[List[str]] = None,
        queries: Optional[List[str]] = None,
        max_length: int = 50,
        contexts: Optional[List[str]] = None,
        suffix: Optional[str] = None,
    ) -> Union[BatchFeature, BatchEncoding]:
        if texts and queries:
            raise ValueError("Only one of 'texts' or 'queries' should be provided.")
        if queries is not None:
            texts = queries
        elif texts is None:
            raise ValueError("No texts or queries provided.")

        if suffix is None:
            suffix = self.query_augmentation_token * 10

        texts = [self.query_prefix + text + suffix for text in texts]
        return self.process_texts(texts=texts)

    @abstractmethod
    def score(
        self,
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        device: Optional[Union[str, torch.device]] = None,
        **kwargs,
    ) -> torch.Tensor:
        pass

    @staticmethod
    def score_single_vector(
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        device: Optional[Union[str, torch.device]] = None,
    ) -> torch.Tensor:
        device = device or _get_torch_device("auto")
        if isinstance(qs, list):
            qs = torch.stack(qs).to(device)
        else:
            qs = qs.to(device)
        if isinstance(ps, list):
            ps = torch.stack(ps).to(device)
        else:
            ps = ps.to(device)
        scores = torch.einsum("bd,cd->bc", qs, ps).to(torch.float32)
        return scores

    @staticmethod
    def score_multi_vector(
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        batch_size: int = 128,
        device: Optional[Union[str, torch.device]] = None,
    ) -> torch.Tensor:
        device = device or _get_torch_device("auto")
        if len(qs) == 0:
            raise ValueError("No queries provided")
        if len(ps) == 0:
            raise ValueError("No passages provided")

        scores_list: List[torch.Tensor] = []
        for i in range(0, len(qs), batch_size):
            qs_batch = torch.nn.utils.rnn.pad_sequence(
                qs[i : i + batch_size], batch_first=True, padding_value=0
            ).to(device)
            scores_batch = []
            for j in range(0, len(ps), batch_size):
                ps_batch = torch.nn.utils.rnn.pad_sequence(
                    ps[j : j + batch_size], batch_first=True, padding_value=0
                ).to(device)
                scores_batch.append(
                    torch.einsum("bnd,csd->bcns", qs_batch, ps_batch).max(dim=3)[0].sum(dim=2)
                )
            scores_list.append(torch.cat(scores_batch, dim=1).cpu())

        return torch.cat(scores_list, dim=0).to(torch.float32)

    @abstractmethod
    def get_n_patches(
        self,
        image_size: Tuple[int, int],
        *args,
        **kwargs,
    ) -> Tuple[int, int]:
        pass


# ==============================================================================
# ColQwen2
# ==============================================================================

class ColQwen2(Qwen2VLModel):
    """
    ColQwen2 model implementation from the "ColPali: Efficient Document Retrieval with Vision
    Language Models" paper.

    Args:
        config (Qwen2VLConfig): The model configuration.
        mask_non_image_embeddings (bool): Whether to ignore all token embeddings except those
            of the image at inference. Defaults to False.
    """

    main_input_name: ClassVar[str] = "doc_input_ids"
    _checkpoint_conversion_mapping = {
        r"^base_model\.model\.custom_text_proj": "custom_text_proj",
        r"^model\.layers": "language_model.layers",
    }

    def __init__(self, config: Qwen2VLConfig, mask_non_image_embeddings: bool = False):
        super().__init__(config=config)

        hidden_size = getattr(self.config, "hidden_size", None)
        if hidden_size is None and hasattr(self.config, "text_config"):
            hidden_size = getattr(self.config.text_config, "hidden_size", None)
        if hidden_size is None:
            raise ValueError(
                f"Unable to determine text hidden size for {type(self.config).__name__}."
            )

        self.dim = 128
        self.custom_text_proj = nn.Linear(hidden_size, self.dim)
        self.padding_side = "left"
        self.mask_non_image_embeddings = mask_non_image_embeddings
        self.post_init()

    @classmethod
    def from_pretrained(cls, *args, **kwargs):
        key_mapping = kwargs.pop("key_mapping", None)
        if key_mapping is None:
            key_mapping = dict(getattr(super(), "_checkpoint_conversion_mapping", {}))
            key_mapping.update(cls._checkpoint_conversion_mapping)
        return super().from_pretrained(*args, **kwargs, key_mapping=key_mapping)

    def forward(self, *args, **kwargs) -> torch.Tensor:
        # Unpad pixel_values produced by ColQwen2Processor before passing to backbone
        if "pixel_values" in kwargs:
            offsets = kwargs["image_grid_thw"][:, 1] * kwargs["image_grid_thw"][:, 2]
            kwargs["pixel_values"] = torch.cat(
                [pv[:off] for pv, off in zip(kwargs["pixel_values"], offsets)],
                dim=0,
            )

        kwargs.pop("return_dict", True)
        kwargs.pop("output_hidden_states", None)
        kwargs.pop("use_cache", None)

        # FIX: Compute RoPE position_ids via get_rope_index() to match encoding notebook.
        # Without this, positional encodings are wrong for text-only queries.
        if "position_ids" not in kwargs or kwargs.get("position_ids") is None:
            input_ids = kwargs["input_ids"]
            # Build mm_token_type_ids: 0=text, 1=image, 2=video
            # For text-only queries all tokens are type 0.
            # For image inputs, mark image_pad tokens as type 1.
            mm_token_type_ids = torch.zeros_like(input_ids, dtype=torch.int)
            if "pixel_values" in kwargs and hasattr(self.config, "image_token_id"):
                mm_token_type_ids[input_ids == self.config.image_token_id] = 1

            position_ids, rope_deltas = self.get_rope_index(
                input_ids=input_ids,
                mm_token_type_ids=mm_token_type_ids,
                image_grid_thw=kwargs.get("image_grid_thw", None),
                video_grid_thw=None,
                attention_mask=kwargs.get("attention_mask", None),
            )
            kwargs["position_ids"] = position_ids

        hidden_states = (
            super()
            .forward(*args, **kwargs, use_cache=False, output_hidden_states=True, return_dict=True)
            .last_hidden_state
        )  # (batch_size, sequence_length, hidden_size)

        proj = self.custom_text_proj(hidden_states)          # (B, S, dim)
        proj = proj / proj.norm(dim=-1, keepdim=True)        # L2 normalise
        proj = proj * kwargs["attention_mask"].unsqueeze(-1)  # zero-out padding

        if "pixel_values" in kwargs and self.mask_non_image_embeddings:
            image_mask = (kwargs["input_ids"] == self.config.image_token_id).unsqueeze(-1)
            proj = proj * image_mask

        return proj

    def forward_with_attentions(self, *args, **kwargs):
        """
        Same as forward() but returns (proj, raw_outputs) so the caller
        can access raw_outputs.attentions for importance scoring.
        """
        if "pixel_values" in kwargs:
            offsets = kwargs["image_grid_thw"][:, 1] * kwargs["image_grid_thw"][:, 2]
            kwargs["pixel_values"] = torch.cat(
                [pv[:off] for pv, off in zip(kwargs["pixel_values"], offsets)],
                dim=0,
            )

        kwargs.pop("return_dict", True)
        kwargs.pop("output_hidden_states", None)
        kwargs.pop("use_cache", None)
        kwargs.pop("output_attentions", None)

        # FIX: Compute RoPE position_ids via get_rope_index()
        if "position_ids" not in kwargs or kwargs.get("position_ids") is None:
            input_ids = kwargs["input_ids"]
            mm_token_type_ids = torch.zeros_like(input_ids, dtype=torch.int)
            if "pixel_values" in kwargs and hasattr(self.config, "image_token_id"):
                mm_token_type_ids[input_ids == self.config.image_token_id] = 1

            position_ids, rope_deltas = self.get_rope_index(
                input_ids=input_ids,
                mm_token_type_ids=mm_token_type_ids,
                image_grid_thw=kwargs.get("image_grid_thw", None),
                video_grid_thw=None,
                attention_mask=kwargs.get("attention_mask", None),
            )
            kwargs["position_ids"] = position_ids

        raw_outputs = (
            super()
            .forward(
                *args, **kwargs,
                use_cache=False,
                output_hidden_states=True,
                output_attentions=True,
                return_dict=True,
            )
        )

        hidden_states = raw_outputs.last_hidden_state
        proj = self.custom_text_proj(hidden_states)
        proj = proj / proj.norm(dim=-1, keepdim=True)
        proj = proj * kwargs["attention_mask"].unsqueeze(-1)

        if "pixel_values" in kwargs and self.mask_non_image_embeddings:
            image_mask = (kwargs["input_ids"] == self.config.image_token_id).unsqueeze(-1)
            proj = proj * image_mask

        return proj, raw_outputs

    @property
    def patch_size(self) -> int:
        return self.visual.config.patch_size

    @property
    def spatial_merge_size(self) -> int:
        return self.visual.config.spatial_merge_size


# ==============================================================================
# ColQwen2Processor
# ==============================================================================

class ColQwen2Processor(BaseVisualRetrieverProcessor, Qwen2VLProcessor):
    """Processor for ColQwen2."""

    visual_prompt_prefix: ClassVar[str] = (
        "<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>"
        "Describe the image.<|im_end|><|endoftext|>"
    )
    query_augmentation_token: ClassVar[str] = "<|endoftext|>"
    image_token: ClassVar[str] = "<|image_pad|>"

    def __init__(
        self,
        image_processor=None,
        tokenizer=None,
        video_processor=None,
        chat_template=None,
        **kwargs,
    ):
        super().__init__(
            image_processor=image_processor,
            tokenizer=tokenizer,
            video_processor=video_processor,
            chat_template=chat_template,
            **kwargs,
        )
        self.tokenizer.padding_side = "left"

    @classmethod
    def from_pretrained(cls, *args, device_map: Optional[str] = None, **kwargs):
        instance = super().from_pretrained(*args, device_map=device_map, **kwargs)
        if "max_num_visual_tokens" in kwargs:
            instance.image_processor.max_pixels = kwargs["max_num_visual_tokens"] * 28 * 28
            instance.image_processor.size["longest_edge"] = instance.image_processor.max_pixels
        return instance

    def process_images(self, images: List[Image.Image]) -> Union[BatchFeature, BatchEncoding]:
        images = [image.convert("RGB") for image in images]
        batch_doc = self(
            text=[self.visual_prompt_prefix] * len(images),
            images=images,
            padding="longest",
            return_tensors="pt",
        )
        # Unpad pixel_values so that DDP / multi-GPU works correctly
        offsets = batch_doc["image_grid_thw"][:, 1] * batch_doc["image_grid_thw"][:, 2]
        pixel_values = list(torch.split(batch_doc["pixel_values"], offsets.tolist()))
        batch_doc["pixel_values"] = torch.nn.utils.rnn.pad_sequence(
            pixel_values, batch_first=True
        )
        return batch_doc

    def process_texts(self, texts: List[str]) -> Union[BatchFeature, BatchEncoding]:
        return self(
            text=texts,
            return_tensors="pt",
            padding="longest",
        )

    def score(
        self,
        qs: List[torch.Tensor],
        ps: List[torch.Tensor],
        device: Optional[Union[str, torch.device]] = None,
        **kwargs,
    ) -> torch.Tensor:
        return self.score_multi_vector(qs, ps, device=device, **kwargs)

    def get_n_patches(
        self,
        image_size: Tuple[int, int],
        spatial_merge_size: int,
    ) -> Tuple[int, int]:
        patch_size = self.image_processor.patch_size
        height_new, width_new = smart_resize(
            width=image_size[0],
            height=image_size[1],
            factor=patch_size * self.image_processor.merge_size,
            min_pixels=self.image_processor.size["shortest_edge"],
            max_pixels=self.image_processor.size["longest_edge"],
        )
        n_patches_x = width_new // patch_size // spatial_merge_size
        n_patches_y = height_new // patch_size // spatial_merge_size
        return n_patches_x, n_patches_y

    def get_image_mask(self, batch_images: BatchFeature) -> torch.Tensor:
        return batch_images.input_ids == self.image_token_id


print("✅ ColQwen2 and ColQwen2Processor class definitions ready.")

✅ ColQwen2 and ColQwen2Processor class definitions ready.


## Cell 4 — Load ColQwen2 Model & Processor

In [5]:
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

print(">>> Loading ColQwen2 model...")
query_model = ColQwen2.from_pretrained(
    COLQWEN2_BASE,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="eager",
)
query_model = PeftModel.from_pretrained(query_model, COLQWEN2_LORA)
query_model.eval()

print(">>> Loading ColQwen2Processor...")
query_processor = ColQwen2Processor.from_pretrained(COLQWEN2_LORA)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ ColQwen2 ready  |  device = {device}")

>>> Loading ColQwen2 model...


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

>>> Loading ColQwen2Processor...
✅ ColQwen2 ready  |  device = cuda


## Cell 5 — Load ViDoRe Pre-encoded PKL Files
 
> **Changed from Notebook 1:**
> The PKL format here comes from Pipeline A of the ViDoRe encoding notebook. Each PKL is a
> `list[dict]` with keys `{embedding, corpus_id, doc_id, page_number_in_doc, domain}`,
> rather than the `(embs, idxs)` / dict-of-arrays format used by the MMDocIR PKLs.
> We build a global index and per-domain lookup tables needed for QA pair construction.

In [6]:
pkl_files = sorted(glob.glob(os.path.join(PAGE_PKL_DIR, "vidore_*.pkl")))
print(f"Found {len(pkl_files)} PKL file(s):")

# Global flat list — index == global_idx
all_page_embeddings: List[np.ndarray] = []

# Parallel metadata list
all_page_meta: List[dict] = []   # {domain, corpus_id, doc_id, page_number_in_doc, global_idx}

for pkl_path in pkl_files:
    with open(pkl_path, "rb") as f:
        items = pickle.load(f)   # list[dict]

    for item in items:
        emb = item["embedding"]
        # Normalise to float32 numpy
        if isinstance(emb, torch.Tensor):
            emb = emb.float().cpu().numpy()
        else:
            emb = np.array(emb, dtype=np.float32)

        global_idx = len(all_page_embeddings)
        all_page_embeddings.append(emb)
        all_page_meta.append({
            "domain":             item.get("domain", "unknown"),
            "corpus_id":          int(item["corpus_id"]),
            "doc_id":             str(item["doc_id"]),
            "page_number_in_doc": int(item["page_number_in_doc"]),
            "global_idx":         global_idx,
        })

    print(f"  Loaded {len(items):>6} embeddings from {os.path.basename(pkl_path)}")

print(f"\nTotal pages : {len(all_page_embeddings)}")
print(f"Dim sample  : {all_page_embeddings[0].shape}")

Found 8 PKL file(s):
  Loaded   1360 embeddings from vidore_computer_science.pkl
  Loaded   2225 embeddings from vidore_energy.pkl
  Loaded   2942 embeddings from vidore_finance_en.pkl
  Loaded   2384 embeddings from vidore_finance_fr.pkl
  Loaded   1110 embeddings from vidore_hr.pkl
  Loaded   5244 embeddings from vidore_industrial.pkl
  Loaded   2313 embeddings from vidore_pharmaceuticals.pkl
  Loaded   1674 embeddings from vidore_physics.pkl

Total pages : 19252
Dim sample  : (755, 128)


## Cell 6 — smart_resize & Patch-Grid Helpers (mirrors encoder)


In [7]:
def _round_by_factor(n, f): return round(n / f) * f
def _ceil_by_factor(n, f):  return math.ceil(n / f) * f
def _floor_by_factor(n, f): return math.floor(n / f) * f


def smart_resize_local(height, width,
                       factor=FACTOR, min_pixels=MIN_PIXELS,
                       max_pixels=MAX_PIXELS, max_ratio=MAX_RATIO):
    """Replicates ColQwen2Retriever.smart_resize."""
    if max(height, width) / min(height, width) > max_ratio:
        raise ValueError(f"Aspect ratio exceeds {max_ratio}")
    h_bar = max(factor, _round_by_factor(height, factor))
    w_bar = max(factor, _round_by_factor(width,  factor))
    if h_bar * w_bar > max_pixels:
        beta  = math.sqrt((height * width) / max_pixels)
        h_bar = _floor_by_factor(height / beta, factor)
        w_bar = _floor_by_factor(width  / beta, factor)
    elif h_bar * w_bar < min_pixels:
        beta  = math.sqrt(min_pixels / (height * width))
        h_bar = _ceil_by_factor(height * beta, factor)
        w_bar = _ceil_by_factor(width  * beta, factor)
    return h_bar, w_bar


def decode_image_field(image_field) -> Image.Image:
    """
    Decode an HF parquet image column value → PIL.Image RGB.
    Pipeline A stores images as dict {'bytes': b'...', 'path': None}.
    """
    if image_field is None:
        raise ValueError("image field is None")
    if isinstance(image_field, Image.Image):
        return image_field.convert("RGB")
    if isinstance(image_field, dict):
        b = image_field.get("bytes")
        if b is None:
            raise ValueError("image dict has no 'bytes' key")
        return Image.open(io.BytesIO(b)).convert("RGB")
    if isinstance(image_field, (bytes, bytearray)):
        return Image.open(io.BytesIO(image_field)).convert("RGB")
    raise TypeError(f"Unknown image field type: {type(image_field)}")


def infer_patch_grid(pil_img: Image.Image,
                     patch_size: int = PATCH_SIZE,
                     merge_size: int = MERGE_SIZE) -> Tuple[int, int]:
    """
    Compute the exact (n_patches_h, n_patches_w) after the ColQwen2 vision tower's
    spatial merge, using the same smart_resize_local() logic that was applied during
    Pipeline A encoding.  Mirrors infer_patch_grid() from the original Notebook 1.
    """
    w, h         = pil_img.size
    h_bar, w_bar = smart_resize_local(h, w)
    return h_bar // patch_size // merge_size, w_bar // patch_size // merge_size


def locate_image_block(emb: np.ndarray, n_expected: int) -> Tuple[int, int]:
    """Centre the image-patch block inside [prefix | image | suffix]."""
    T       = emb.shape[0]
    n       = min(n_expected, T)
    leftover = T - n
    start   = leftover // 2
    return start, start + n


print("✅ smart_resize helpers ready.")

✅ smart_resize helpers ready.


## Cell 7 — Load Corpus Images & Compute Exact Per-Page Patch Grids
 
> **Restored to Notebook 1 approach:**
> The ViDoRe corpus parquets (inside `VIDORE_ROOT`) contain the same `image` column that
> Pipeline A used during encoding. We load them here, decode each page's PIL image, and
> compute exact `(H, W)` patch grids via `smart_resize_local()` + `infer_patch_grid()` —
> identical to how Notebook 1 handled MMDocIR. This gives pool2d and hier the correct
> spatial structure instead of a rough token-count approximation.

In [8]:
# ── Helper: find corpus parquet paths for one domain ──────────────────────────

def find_corpus_parquets(vidore_root: str, domain_name: str) -> List[str]:
    """Return sorted list of corpus parquet paths for a given domain."""
    outer = os.path.join(vidore_root, f"vidore_v3_{domain_name}")
    candidates = [
        os.path.join(outer, f"vidore_v3_{domain_name}", "corpus"),
        os.path.join(outer, "corpus"),
    ]
    corpus_dir = next((c for c in candidates if os.path.isdir(c)), None)
    if corpus_dir is None:
        raise FileNotFoundError(
            f"Corpus dir not found for domain '{domain_name}' under {vidore_root}"
        )
    files = sorted(glob.glob(os.path.join(corpus_dir, "*.parquet")))
    if not files:
        raise FileNotFoundError(f"No parquet files in {corpus_dir}")
    return files


# ── Pre-allocate output arrays, indexed parallel to all_page_embeddings ────────

grids:  List[Tuple[int, int]]           = [(0, 0)] * len(all_page_embeddings)
slices: List[Optional[Tuple[int, int]]] = [None]   * len(all_page_embeddings)

# Build a per-domain index: domain → list of (global_position, corpus_id)
# so we can fill grids/slices in the right positions without holding all images.
from collections import defaultdict
domain_positions: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
for pos, meta in enumerate(all_page_meta):
    domain_positions[meta["domain"]].append((pos, meta["corpus_id"]))

domains_in_pkls = sorted(domain_positions.keys())
print(f"Domains to process: {domains_in_pkls}\n")

n_exact    = 0
n_fallback = 0

def _fallback_grid(emb: np.ndarray) -> Tuple[int, int]:
    """Token-count estimate of (H, W) — used only when image is unavailable."""
    n_img = max(1, emb.shape[0] - 4)
    sq    = int(math.isqrt(n_img))
    while sq > 1 and n_img % sq != 0:
        sq -= 1
    return sq, (n_img // sq if sq > 0 else n_img)


# ── Main loop: one domain at a time ───────────────────────────────────────────

for domain_name in domains_in_pkls:
    positions = domain_positions[domain_name]   # [(global_pos, corpus_id), ...]

    # --- load this domain's corpus parquet (image column only) ---------------
    try:
        parquet_files = find_corpus_parquets(VIDORE_ROOT, domain_name)
    except FileNotFoundError as e:
        print(f"  ⚠️  {e} — falling back to token-count grids for all {len(positions)} pages")
        for pos, _ in positions:
            H, W  = _fallback_grid(all_page_embeddings[pos])
            n_img = H * W
            s, e  = locate_image_block(all_page_embeddings[pos], n_img)
            if (e - s) != n_img:
                n_img = e - s; H, W = 1, n_img
            grids[pos]  = (H, W)
            slices[pos] = (s, e)
            n_fallback += 1
        continue

    # Read only corpus_id + image — discard everything else immediately
    corpus_df = pd.concat(
        [pd.read_parquet(p, columns=["corpus_id", "image"]) for p in parquet_files],
        ignore_index=True,
    )

    # Build a lightweight corpus_id → (H, W) grid map for this domain only.
    # We decode each image, compute the grid, then drop the PIL object immediately.
    grid_map: Dict[int, Tuple[int, int]] = {}
    n_ok = 0
    for _, row in corpus_df.iterrows():
        cid = int(row["corpus_id"])
        try:
            pil        = decode_image_field(row["image"])
            grid_map[cid] = infer_patch_grid(pil)
            del pil        # free decoded image right away
            n_ok += 1
        except Exception as ex:
            print(f"  ⚠️  {domain_name} corpus_id={cid} decode error: {ex}")

    # Free the entire corpus DataFrame — images are the heaviest part
    del corpus_df
    gc.collect()

    # --- fill grids/slices for every page in this domain --------------------
    for pos, cid in positions:
        emb = all_page_embeddings[pos]
        if cid in grid_map:
            H, W = grid_map[cid]
            n_exact += 1
        else:
            H, W = _fallback_grid(emb)
            n_fallback += 1

        n_img = H * W
        s, e  = locate_image_block(emb, n_img)
        if (e - s) != n_img:
            n_img = e - s; H, W = 1, n_img
        grids[pos]  = (H, W)
        slices[pos] = (s, e)

    del grid_map
    gc.collect()
    print(f"  {domain_name:25s}: {n_ok}/{len(positions)} exact grids")

# ── Summary ───────────────────────────────────────────────────────────────────

total_tokens = sum(e.shape[0] for e in all_page_embeddings)
print(f"\nTotal tokens across all pages = {total_tokens}")
if NEED_GRID:
    total_img_tokens = sum(H * W for H, W in grids)
    print(f"Image tokens  = {total_img_tokens}  |  Text tokens = {total_tokens - total_img_tokens}")
    print(f"Exact grids   = {n_exact}  |  Fallback (token-count) = {n_fallback}")
print("✅ Patch grids ready.")

Domains to process: ['computer_science', 'energy', 'finance_en', 'finance_fr', 'hr', 'industrial', 'pharmaceuticals', 'physics']

  computer_science         : 1360/1360 exact grids
  energy                   : 2225/2225 exact grids
  finance_en               : 2942/2942 exact grids
  finance_fr               : 2384/2384 exact grids
  hr                       : 1110/1110 exact grids
  industrial               : 5244/5244 exact grids
  pharmaceuticals          : 2313/2313 exact grids
  physics                  : 1674/1674 exact grids

Total tokens across all pages = 14470118
Image tokens  = 14258346  |  Text tokens = 211772
Exact grids   = 19252  |  Fallback (token-count) = 0
✅ Patch grids ready.


## Cell 8 — Compression Strategy Functions

In [9]:
# ── Random pruning ─────────────────────────────────────────────────────────────

def random_prune(emb: np.ndarray, keep_ratio: float,
                 rng: np.random.Generator) -> np.ndarray:
    N = emb.shape[0]
    k = max(1, int(round(N * keep_ratio)))
    if k >= N:
        return emb.copy()
    idx = np.sort(rng.choice(N, size=k, replace=False))
    return emb[idx]


# ── 1-D average pooling ────────────────────────────────────────────────────────

def pool_1d(emb: np.ndarray, keep_ratio: float) -> np.ndarray:
    N = emb.shape[0]
    k = max(1, int(round(N * keep_ratio)))
    if k >= N:
        return emb.copy()
    edges = np.linspace(0, N, num=k + 1, dtype=int)
    out   = np.empty((k, emb.shape[1]), dtype=emb.dtype)
    for i in range(k):
        s, e = edges[i], max(edges[i] + 1, edges[i + 1])
        out[i] = emb[s:e].mean(axis=0)
    norms = np.linalg.norm(out, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return out / norms


# ── 2-D average pooling ────────────────────────────────────────────────────────

def _pick_2d_block(H: int, W: int, keep_ratio: float) -> Tuple[int, int]:
    N      = H * W
    target = max(1, int(round(N * keep_ratio)))
    if target >= N:
        return 1, 1
    best     = None
    max_side = max(2, int(math.ceil(math.sqrt(N / target))) + 2)
    for bh in range(1, max_side + 1):
        for bw in range(1, max_side + 1):
            if bh == 1 and bw == 1:
                continue
            out_n = math.ceil(H / bh) * math.ceil(W / bw)
            score = (abs(out_n - target), abs(bh - bw))
            if best is None or score < best[0]:
                best = (score, bh, bw)
    return best[1], best[2]


def pool_2d(img_emb: np.ndarray, keep_ratio: float,
            grid: Tuple[int, int]) -> np.ndarray:
    H, W = grid
    assert img_emb.shape[0] == H * W, \
        f"image-token count {img_emb.shape[0]} != H*W = {H*W}"
    if keep_ratio >= 1.0:
        return img_emb.copy()
    bh, bw = _pick_2d_block(H, W, keep_ratio)
    D      = img_emb.shape[1]
    grid_e = img_emb.reshape(H, W, D)
    out_h  = math.ceil(H / bh)
    out_w  = math.ceil(W / bw)
    out    = np.empty((out_h, out_w, D), dtype=img_emb.dtype)
    for i in range(out_h):
        for j in range(out_w):
            r0, r1 = i * bh, min((i + 1) * bh, H)
            c0, c1 = j * bw, min((j + 1) * bw, W)
            out[i, j] = grid_e[r0:r1, c0:c1].reshape(-1, D).mean(axis=0)
    out   = out.reshape(out_h * out_w, D)
    norms = np.linalg.norm(out, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return out / norms


# ── Helpers shared between strategies ─────────────────────────────────────────

def _get_scope(emb: np.ndarray,
               sl: Optional[Tuple[int, int]],
               compress_what: str = "all_tokens"):
    """Split embedding into (prefix, middle, suffix) for compression."""
    if compress_what == "all_tokens" or sl is None:
        return emb[:0], emb, emb[:0]
    s, e = sl
    return emb[:s], emb[s:e], emb[e:]


# ── Hierarchical Ward pooling ──────────────────────────────────────────────────

def ward_pool_torch(vecs: torch.Tensor, n_clusters: int) -> torch.Tensor:
    N = vecs.shape[0]
    if N <= n_clusters:
        return vecs.clone()
    device = vecs.device
    sums   = vecs.clone().float()
    sizes  = torch.ones(N, device=device, dtype=torch.float32)
    cents  = F.normalize(sums, dim=-1)
    sim    = (cents @ cents.t()).clamp_(-1.0, 1.0)
    dist   = 2.0 - 2.0 * sim
    ni = sizes.unsqueeze(1); nj = sizes.unsqueeze(0)
    ward   = (ni * nj) / (ni + nj) * dist
    tril_mask = torch.ones(N, N, dtype=torch.bool, device=device).tril()
    ward.masked_fill_(tril_mask, float('inf'))
    INF = float('inf')
    C   = N
    while C > n_clusters:
        flat_idx = int(ward[:C, :C].argmin().item())
        i = flat_idx // C; j = flat_idx % C
        if i > j: i, j = j, i
        sums[i]  = sums[i]  + sums[j]
        sizes[i] = sizes[i] + sizes[j]
        cents[i] = F.normalize(sums[i:i+1], dim=-1).squeeze(0)
        last = C - 1
        if j != last:
            moved = ward[:last, last].clone()
            sums[j]  = sums[last];  sizes[j] = sizes[last];  cents[j] = cents[last]
            if j > 0:          ward[:j, j]      = moved[:j]
            if j + 1 < last:   ward[j, j+1:last] = moved[j+1:last]
            ward[j, j] = INF
        new_C   = last
        sim_i   = cents[:new_C] @ cents[i]
        sim_i.clamp_(-1.0, 1.0)
        dist_i  = 2.0 - 2.0 * sim_i
        ni_i    = sizes[i]; nj_all = sizes[:new_C]
        ward_i  = (ni_i * nj_all) / (ni_i + nj_all) * dist_i
        if i + 1 < new_C: ward[i, i+1:new_C] = ward_i[i+1:new_C]
        if i > 0:          ward[:i, i]        = ward_i[:i]
        ward[i, i] = INF
        C = new_C
    return cents[:C]


def ward_pool_batched(vecs_list: List[torch.Tensor],
                      K_list: List[int], device) -> List[torch.Tensor]:
    B     = len(vecs_list)
    N_list = [v.shape[0] for v in vecs_list]
    D      = vecs_list[0].shape[1]
    N_max  = max(N_list)
    sums   = torch.zeros(B, N_max, D, device=device, dtype=torch.float32)
    sizes  = torch.zeros(B, N_max,    device=device, dtype=torch.float32)
    for b, v in enumerate(vecs_list):
        n = N_list[b]; sums[b, :n] = v.float(); sizes[b, :n] = 1.0
    cents    = F.normalize(sums + 1e-30, dim=-1)
    row_mask = sizes > 0
    cents    = cents * row_mask.unsqueeze(-1)
    C = torch.tensor(N_list, device=device, dtype=torch.long)
    K = torch.tensor(K_list, device=device, dtype=torch.long)
    arange    = torch.arange(N_max, device=device)
    base_tril = (arange.unsqueeze(1) >= arange.unsqueeze(0))
    b_idx     = torch.arange(B, device=device)
    INF       = float('inf')

    def _build_ward():
        sim  = torch.bmm(cents, cents.transpose(1, 2)).clamp_(-1.0, 1.0)
        dist = 2.0 - 2.0 * sim
        ni = sizes.unsqueeze(2); nj = sizes.unsqueeze(1)
        w  = (ni * nj) / (ni + nj).clamp(min=1e-30) * dist
        w.masked_fill_(base_tril.unsqueeze(0), INF)
        valid   = arange.unsqueeze(0) < C.unsqueeze(1)
        invalid = ~(valid.unsqueeze(2) & valid.unsqueeze(1))
        w.masked_fill_(invalid, INF)
        return w

    max_steps = max(N_list[b] - K_list[b] for b in range(B))
    for _ in range(max_steps):
        if bool((C <= K).all().item()):
            break
        done_mask = C <= K
        ward      = _build_ward()
        flat      = ward.view(B, -1)
        best      = flat.argmin(dim=1)
        i_all     = torch.minimum(best // N_max, best % N_max)
        j_all     = torch.maximum(best // N_max, best % N_max)
        i_all = torch.where(done_mask, torch.zeros_like(i_all), i_all)
        j_all = torch.where(done_mask, torch.zeros_like(j_all), j_all)
        sums_j  = sums[b_idx, j_all]  * (~done_mask).unsqueeze(-1).float()
        sizes_j = sizes[b_idx, j_all] * (~done_mask).float()
        sums_i_new  = sums[b_idx, i_all]  + sums_j
        sizes_i_new = sizes[b_idx, i_all] + sizes_j
        sums[b_idx, i_all]  = sums_i_new
        sizes[b_idx, i_all] = sizes_i_new
        cents[b_idx, i_all] = F.normalize(sums_i_new + 1e-30, dim=-1)
        last      = C - 1
        do_swap   = (~done_mask) & (j_all != last)
        do_swap_f = do_swap.unsqueeze(-1).float()
        sums[b_idx, j_all]  = sums[b_idx, last]  * do_swap_f + sums[b_idx, j_all]  * (1 - do_swap_f)
        cents[b_idx, j_all] = cents[b_idx, last]  * do_swap_f + cents[b_idx, j_all] * (1 - do_swap_f)
        sizes[b_idx, j_all] = torch.where(do_swap, sizes[b_idx, last], sizes[b_idx, j_all])
        C = torch.where(done_mask, C, C - 1)
        new_valid = arange.unsqueeze(0) < C.unsqueeze(1)
        sums  = sums  * new_valid.unsqueeze(-1).float()
        cents = cents * new_valid.unsqueeze(-1).float()
        sizes = sizes * new_valid.float()

    C_cpu = C.cpu().tolist()
    return [cents[b, :C_cpu[b]].clone() for b in range(B)]


def hier_ward_compress_batch_all_ratios(
    pages: List[np.ndarray],
    ratios: List[float],
    device: str = "cuda",
) -> Dict[float, List[np.ndarray]]:
    dev_str = device if (device != "cuda" or torch.cuda.is_available()) else "cpu"
    dev     = torch.device(dev_str)
    prev_tensors = [
        F.normalize(torch.from_numpy(p).float(), dim=-1).to(dev)
        for p in pages
    ]
    Ns        = [p.shape[0] for p in pages]
    out_dtypes = [p.dtype   for p in pages]
    results: Dict[float, List[np.ndarray]] = {r: [None]*len(pages) for r in ratios}

    for ratio in sorted(ratios, reverse=True):
        Ks          = [max(1, int(round(N * ratio))) for N in Ns]
        to_pool_idx = [i for i, (t, k) in enumerate(zip(prev_tensors, Ks)) if t.shape[0] > k]
        if to_pool_idx:
            if dev.type == "cuda":
                sub_pages = [prev_tensors[i] for i in to_pool_idx]
                sub_K     = [Ks[i]           for i in to_pool_idx]
                sub_out   = ward_pool_batched(sub_pages, sub_K, dev)
                for i, o in zip(to_pool_idx, sub_out):
                    prev_tensors[i] = o
            else:
                for i in to_pool_idx:
                    prev_tensors[i] = ward_pool_torch(prev_tensors[i], Ks[i])
        for i, t in enumerate(prev_tensors):
            results[ratio][i] = t.cpu().numpy().astype(out_dtypes[i])
    return results


# ── On-the-fly compression dispatcher ─────────────────────────────────────────

def compress_embeddings_inplace(
    encoded_pages: List[np.ndarray],
    strategy: str,
    ratio: float,
    grids: List[Tuple[int, int]],
    slices: List[Optional[Tuple[int, int]]],
    rng: np.random.Generator,
    hier_device: str = HIER_DEVICE,
    hier_batch_size: int = HIER_BATCH_SIZE,
) -> List[np.ndarray]:
    """
    Apply compression to all pages on-the-fly and return the compressed list.
    For 'hier', all ratios are handled externally; pass a single ratio here.
    """
    out_pages: List[np.ndarray] = []

    if strategy == "hier":
        dev_str = hier_device if (hier_device != "cuda" or torch.cuda.is_available()) else "cpu"
        # Build middle arrays (for COMPRESS_WHAT == "all_tokens", middle == full emb)
        middles  = [emb for emb in encoded_pages]   # all_tokens: compress full sequence
        prefixes = [emb[:0] for emb in encoded_pages]
        suffixes = [emb[:0] for emb in encoded_pages]

        n_pages = len(middles)
        out_per_ratio = {ratio: [None] * n_pages}
        for batch_start in range(0, n_pages, hier_batch_size):
            batch_end = min(batch_start + hier_batch_size, n_pages)
            batch_mid = middles[batch_start:batch_end]
            pooled_by_ratio = hier_ward_compress_batch_all_ratios(
                batch_mid, [ratio], device=dev_str,
            )
            for local_i, pooled in enumerate(pooled_by_ratio[ratio]):
                global_i = batch_start + local_i
                out_per_ratio[ratio][global_i] = np.concatenate(
                    [prefixes[global_i], pooled, suffixes[global_i]], axis=0
                )
        return out_per_ratio[ratio]

    # Non-hierarchical strategies: random / pool1d / pool2d
    for emb, (H, W), sl in zip(encoded_pages, grids, slices):
        if strategy == "pool2d":
            # pool2d always operates on image block only
            assert sl is not None, "pool2d requires grid inference"
            s, e = sl
            prefix, img, suffix = emb[:s], emb[s:e], emb[e:]
            mid_c = pool_2d(img, ratio, grid=(H, W))
        else:
            prefix, mid, suffix = _get_scope(emb, sl, compress_what="all_tokens")
            if strategy == "random":
                mid_c = random_prune(mid, ratio, rng)
            elif strategy == "pool1d":
                mid_c = pool_1d(mid, ratio)
            else:
                raise ValueError(f"Unknown strategy: {strategy}")
        out_pages.append(np.concatenate([prefix, mid_c, suffix], axis=0))

    return out_pages


print("✅ Compression strategy functions ready.")

✅ Compression strategy functions ready.


## Cell 9 — Core Evaluation Utilities (MaxSim, Metrics, QA Pairs)
 
> **Changed from Notebook 1:**
> QA pair construction is completely rewritten to use ViDoRe V3 queries and qrels parquets
> instead of MMDocIR annotations JSONL. The ViDoRe dataset has one folder per domain, each
> with `queries/`, `qrels/`, and `corpus/` sub-directories. We discover all domains from
> `VIDORE_ROOT`, load query text + graded relevance scores (score ∈ {1, 2}), and build
> QA pairs with global/local index mappings compatible with the existing MaxSim evaluation loop.
 

In [10]:
# ── Build padded doc matrix ────────────────────────────────────────────────────

def build_doc_matrix(embeddings: List[np.ndarray], device: str):
    arrays  = [torch.from_numpy(e).float() for e in embeddings]
    max_len = max(a.shape[0] for a in arrays)
    D       = arrays[0].shape[1]
    n       = len(arrays)
    mat     = torch.zeros(n, max_len, D, dtype=torch.float32)
    mask    = torch.zeros(n, max_len, dtype=torch.bool)
    for i, a in enumerate(arrays):
        L = a.shape[0]
        mat[i, :L] = F.normalize(a, dim=-1)
        mask[i, :L] = True
    return mat.to(device), mask.to(device)


# ── MaxSim ─────────────────────────────────────────────────────────────────────

@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values   # (N_q, n_docs)


# ── Metrics ────────────────────────────────────────────────────────────────────

def recall(retrieved, gt):
    return len(set(retrieved) & set(gt)) / len(gt) if gt else 0

def ndcg(retrieved, gt):
    ideal = sum(1/np.log2(i+2) for i in range(min(len(retrieved), len(gt))))
    actual = sum(1/np.log2(i+2) for i, r in enumerate(retrieved) if r in gt)
    return actual / ideal if ideal > 0 else 0

def top_k_indices(scores, k):
    return [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:k]]

def hit_metrics(top_indices, gt_local, topk_list=TOPK_EVAL):
    return {
        **{f'r{k}':    recall(top_indices[:k], gt_local) for k in topk_list},
        **{f'n{k}':    ndcg(  top_indices[:k], gt_local) for k in topk_list},
    }

def _init_metric():
    m = {'count': 0}
    for k in TOPK_EVAL: m[f'r{k}'] = 0.0; m[f'n{k}'] = 0.0
    return m

def _add_metric(dst, src):
    for k in TOPK_EVAL:
        dst[f'r{k}'] += float(src[f'r{k}'])
        dst[f'n{k}'] += float(src[f'n{k}'])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(all_metrics, all_domain_metrics, key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    if domain not in all_domain_metrics:
        all_domain_metrics[domain] = {}
    _add_metric(_ensure(all_domain_metrics[domain], key), m)

def print_summary(all_metrics, method_keys, title=""):
    if title: print(f"\n{'='*60}\n{title}\n{'='*60}")
    header = f"{'Method':<40}"
    for k in TOPK_EVAL: header += f" {'R@'+str(k):>7}"
    for k in TOPK_EVAL: header += f" {'nDCG@'+str(k):>9}"
    print(header)
    print("-" * (40 + 7*len(TOPK_EVAL) + 9*len(TOPK_EVAL)))
    for key in method_keys:
        if key not in all_metrics: continue
        m   = all_metrics[key]; cnt = m['count'] or 1
        row = f"{key:<40}"
        for k in TOPK_EVAL: row += f" {m[f'r{k}']/cnt*100:6.2f}%"
        for k in TOPK_EVAL: row += f" {m[f'n{k}']/cnt:8.4f}"
        print(row)


# ── Discover ViDoRe domains & load queries + qrels ────────────────────────────

def discover_vidore_domains(vidore_root: str) -> List[dict]:
    """
    Scan VIDORE_ROOT for vidore_v3_{domain} folders.
    Returns list of dicts: {name, queries_files, qrels_files}
    """
    domains = []
    for outer in sorted(glob.glob(os.path.join(vidore_root, "vidore_v3_*"))):
        if not os.path.isdir(outer):
            continue
        domain_name = os.path.basename(outer).replace("vidore_v3_", "")
        # Handle double-nested layout: vidore_v3_X/vidore_v3_X/queries
        candidates = [
            os.path.join(outer, os.path.basename(outer)),
            outer,
        ]
        base = next((c for c in candidates
                     if os.path.isdir(os.path.join(c, "queries"))), None)
        if base is None:
            print(f"  ⚠️  {domain_name}: no queries folder found — skip")
            continue
        queries_files = sorted(glob.glob(os.path.join(base, "queries", "*.parquet")))
        qrels_files   = sorted(glob.glob(os.path.join(base, "qrels",   "*.parquet")))
        if not (queries_files and qrels_files):
            print(f"  ⚠️  {domain_name}: missing queries or qrels parquets — skip")
            continue
        domains.append({
            "name":          domain_name,
            "queries_files": queries_files,
            "qrels_files":   qrels_files,
        })
    return domains


# ── Build lookup tables ────────────────────────────────────────────────────────

# (domain, corpus_id) → global embedding index
domain_corpusid_to_global: Dict[Tuple[str, int], int] = {
    (m["domain"], m["corpus_id"]): m["global_idx"]
    for m in all_page_meta
}

# domain → sorted list of global embedding indices (defines "local" ordering)
domain_global_indices: Dict[str, List[int]] = {}
for m in all_page_meta:
    domain_global_indices.setdefault(m["domain"], []).append(m["global_idx"])
for dom in domain_global_indices:
    domain_global_indices[dom] = sorted(domain_global_indices[dom])


# ── Build QA pairs ─────────────────────────────────────────────────────────────

print(f"Discovering ViDoRe domains from: {VIDORE_ROOT}")
vidore_domains = discover_vidore_domains(VIDORE_ROOT)
print(f"Found {len(vidore_domains)} domain(s): {[d['name'] for d in vidore_domains]}")

qa_pairs = []
n_dropped = 0

for d in vidore_domains:
    dname = d["name"]
    if dname not in domain_global_indices:
        print(f"  ⚠️  {dname}: no encoded pages found — skip")
        continue

    q_df  = pd.concat([pd.read_parquet(f) for f in d["queries_files"]], ignore_index=True)
    qr_df = pd.concat([pd.read_parquet(f) for f in d["qrels_files"]],  ignore_index=True)

    # Keep only relevant rows (score 1 = relevant, 2 = highly relevant)
    qr_df = qr_df[qr_df["score"].isin([1, 2])]

    # query_id → list of global embedding indices
    local_gt_map: Dict[int, List[int]] = {}
    for _, qr_row in qr_df.iterrows():
        qid = int(qr_row["query_id"])
        cid = int(qr_row["corpus_id"])
        key = (dname, cid)
        if key in domain_corpusid_to_global:
            local_gt_map.setdefault(qid, []).append(domain_corpusid_to_global[key])

    # global → local index within this domain
    dom_globals    = domain_global_indices[dname]
    global_to_local = {g: l for l, g in enumerate(dom_globals)}

    for _, row in q_df.iterrows():
        qid        = int(row["query_id"])
        gt_globals = list(set(local_gt_map.get(qid, [])))
        gt_locals  = [global_to_local[g] for g in gt_globals if g in global_to_local]
        if not gt_locals:
            n_dropped += 1
            continue
        qa_pairs.append({
            "question":          str(row["query"]),
            "gt_local_indices":  gt_locals,
            "doc_embed_indices": dom_globals,
            "doc_name":          dname,
            "domain":            dname,
        })

qa_pairs = [q for q in qa_pairs if q["gt_local_indices"]]
print(f"\nTotal QA pairs : {len(qa_pairs)}")
print(f"Dropped (no GT): {n_dropped}")
print("✅ Evaluation utilities ready.")

Discovering ViDoRe domains from: /kaggle/input/datasets/namthi/vidore-v3
Found 8 domain(s): ['computer_science', 'energy', 'finance_en', 'finance_fr', 'hr', 'industrial', 'pharmaceuticals', 'physics']

Total QA pairs : 14514
Dropped (no GT): 0
✅ Evaluation utilities ready.


## Cell 10 — Build Baseline Doc Matrix

In [11]:
print("Building baseline (full-token) doc matrix...")
baseline_doc_matrix, baseline_doc_mask = build_doc_matrix(all_page_embeddings, device)
print(f"Baseline doc matrix shape: {baseline_doc_matrix.shape}")

Building baseline (full-token) doc matrix...
Baseline doc matrix shape: torch.Size([19252, 779, 128])


## Cell 11 — Main Evaluation Loop

Evaluates the **baseline** plus every **(strategy, ratio)** combination by compressing
document embeddings on-the-fly and computing MaxSim retrieval metrics.

In [12]:
# ── Method keys ───────────────────────────────────────────────────────────────
STRATEGY_RATIO_KEYS = (
    ["baseline"] +
    [f"{s}_r{int(r*100):02d}" for s in STRATEGIES for r in KEEP_RATIOS]
)

all_metrics        = {}
all_domain_metrics = {}
query_rows         = []

# ── Encode all queries once ───────────────────────────────────────────────────
print("Encoding all queries...")
q_norms = []
for item in tqdm(qa_pairs, desc="Query encoding"):
    q_inputs = query_processor.process_queries(
        queries=[item["question"]]
    ).to(device)
    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)
    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norms.append(F.normalize(q_emb, dim=-1))

print(f"Encoded {len(q_norms)} queries.")

# ── Baseline: full-token evaluation ───────────────────────────────────────────
print("\n>>> BASELINE: Full-token MaxSim")
for q_idx, (item, q_norm) in enumerate(tqdm(
        zip(qa_pairs, q_norms), total=len(qa_pairs), desc="Baseline")):
    doc_idxs = item['doc_embed_indices']
    gt_local  = item['gt_local_indices']
    domain    = item['domain']

    doc_mat  = baseline_doc_matrix[doc_idxs]
    doc_msk  = baseline_doc_mask[doc_idxs]
    scores   = fast_maxsim(q_norm, doc_mat, doc_msk).sum(dim=0).cpu().tolist()
    m        = hit_metrics(top_k_indices(scores, max(TOPK_EVAL)), gt_local)
    record(all_metrics, all_domain_metrics, 'baseline', m, domain)

    row = {'query_id': q_idx, 'doc_name': item['doc_name'],
           'domain': domain, 'question': item['question']}
    for k in TOPK_EVAL:
        row[f'baseline_r@{k}']    = round(m[f'r{k}'], 4)
        row[f'baseline_ndcg@{k}'] = round(m[f'n{k}'], 4)
    query_rows.append(row)

print_summary(all_metrics, ['baseline'], title="Baseline Results")

# ── Compressed evaluations: each (strategy, ratio) ────────────────────────────
for strategy in STRATEGIES:
    print(f"\n>>> Strategy: {strategy.upper()}")

    # Pre-compute hier compression for all ratios at once (more efficient)
    if strategy == "hier":
        dev_str = HIER_DEVICE if (HIER_DEVICE != "cuda" or torch.cuda.is_available()) else "cpu"
        print(f"  Pre-computing hierarchical Ward pooling on {dev_str} ...")

        middles = [emb for emb in all_page_embeddings]   # all_tokens
        n_pages = len(middles)

        # pooled_by_ratio[ratio] = List[np.ndarray]
        pooled_by_ratio: Dict[float, List[np.ndarray]] = {r: [None]*n_pages for r in KEEP_RATIOS}

        for batch_start in tqdm(range(0, n_pages, HIER_BATCH_SIZE),
                                 desc=f"hier batches (size={HIER_BATCH_SIZE})"):
            batch_end = min(batch_start + HIER_BATCH_SIZE, n_pages)
            batch_mid = middles[batch_start:batch_end]
            batch_out = hier_ward_compress_batch_all_ratios(batch_mid, KEEP_RATIOS, device=dev_str)
            for r in KEEP_RATIOS:
                for local_i, arr in enumerate(batch_out[r]):
                    pooled_by_ratio[r][batch_start + local_i] = arr

        hier_doc_matrices: Dict[float, Tuple] = {}
        for r in KEEP_RATIOS:
            hier_doc_matrices[r] = build_doc_matrix(pooled_by_ratio[r], device)
            tokens_after = sum(p.shape[0] for p in pooled_by_ratio[r])
            pct = int(round(r * 100))
            print(f"  [hier @ {pct:2d}%] {total_tokens} -> {tokens_after} tokens")

    for ratio in KEEP_RATIOS:
        pct = int(round(ratio * 100))
        key = f"{strategy}_r{pct:02d}"
        print(f"\n  Evaluating {key}...")

        # ── Build compressed doc matrix ───────────────────────────────────
        if strategy == "hier":
            comp_doc_matrix, comp_doc_mask = hier_doc_matrices[ratio]
        else:
            rng = np.random.default_rng(SEED)
            compressed_pages = compress_embeddings_inplace(
                all_page_embeddings, strategy, ratio,
                grids, slices, rng,
                hier_device=HIER_DEVICE,
                hier_batch_size=HIER_BATCH_SIZE,
            )
            tokens_after = sum(p.shape[0] for p in compressed_pages)
            print(f"    {total_tokens} -> {tokens_after} tokens "
                  f"(actual ratio = {tokens_after/total_tokens:.3f})")
            comp_doc_matrix, comp_doc_mask = build_doc_matrix(compressed_pages, device)

        # ── Score every query against its document's compressed pages ─────
        for q_idx, (item, q_norm) in enumerate(zip(qa_pairs, q_norms)):
            doc_idxs = item['doc_embed_indices']
            gt_local  = item['gt_local_indices']
            domain    = item['domain']

            doc_mat = comp_doc_matrix[doc_idxs]
            doc_msk = comp_doc_mask[doc_idxs]
            scores  = fast_maxsim(q_norm, doc_mat, doc_msk).sum(dim=0).cpu().tolist()
            m       = hit_metrics(top_k_indices(scores, max(TOPK_EVAL)), gt_local)
            record(all_metrics, all_domain_metrics, key, m, domain)

            row = query_rows[q_idx]
            for k in TOPK_EVAL:
                row[f'{key}_r@{k}']    = round(m[f'r{k}'], 4)
                row[f'{key}_ndcg@{k}'] = round(m[f'n{k}'], 4)

        # Free GPU memory before next iteration
        del comp_doc_matrix, comp_doc_mask
        torch.cuda.empty_cache()

        print_summary(all_metrics, [key])

    # Free hier matrices after finishing this strategy
    if strategy == "hier":
        del hier_doc_matrices
        torch.cuda.empty_cache()

print("\n✅ All evaluations complete.")

Encoding all queries...


Query encoding:   0%|          | 0/14514 [00:00<?, ?it/s]

Encoded 14514 queries.

>>> BASELINE: Full-token MaxSim


Baseline:   0%|          | 0/14514 [00:00<?, ?it/s]


Baseline Results
Method                                       R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
--------------------------------------------------------------------------------------------------------
baseline                                  19.40%  33.50%  40.79%  50.88%   0.4262   0.4194   0.4277   0.4515

>>> Strategy: RANDOM

  Evaluating random_r10...
    14470118 -> 1453452 tokens (actual ratio = 0.100)
Method                                       R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
--------------------------------------------------------------------------------------------------------
random_r10                                13.89%  24.76%  30.92%  39.73%   0.3278   0.3176   0.3254   0.3453

  Evaluating random_r20...
    14470118 -> 2892232 tokens (actual ratio = 0.200)
Method                                       R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
-----------------------------

hier batches (size=1024):   0%|          | 0/19 [00:00<?, ?it/s]

  [hier @ 10%] 14470118 -> 1453452 tokens
  [hier @ 20%] 14470118 -> 2892232 tokens
  [hier @ 30%] 14470118 -> 4335045 tokens
  [hier @ 40%] 14470118 -> 5788479 tokens
  [hier @ 50%] 14470118 -> 7244414 tokens
  [hier @ 60%] 14470118 -> 8681639 tokens
  [hier @ 70%] 14470118 -> 10124434 tokens
  [hier @ 80%] 14470118 -> 11577886 tokens
  [hier @ 90%] 14470118 -> 13027305 tokens

  Evaluating hier_r10...
Method                                       R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
--------------------------------------------------------------------------------------------------------
hier_r10                                  14.44%  25.78%  31.86%  41.35%   0.3278   0.3237   0.3319   0.3561

  Evaluating hier_r20...
Method                                       R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
--------------------------------------------------------------------------------------------------------
hier_r20              

## Cell 12 — Save Results to CSV

In [13]:
# ── Build Summary DataFrame ────────────────────────────────────────────────────

summary_rows = []
for key in STRATEGY_RATIO_KEYS:
    if key not in all_metrics:
        continue
    m   = all_metrics[key]
    cnt = m['count'] or 1
    if key == 'baseline':
        strategy_name = 'baseline'; ratio_val = 1.0; pct_val = 100
    else:
        parts         = key.rsplit('_r', 1)
        strategy_name = parts[0]
        pct_val       = int(parts[1])
        ratio_val     = pct_val / 100.0
    row = {
        'method':     key,
        'strategy':   strategy_name,
        'keep_ratio': ratio_val,
        'keep_pct':   pct_val,
        'n_queries':  cnt,
    }
    for k in TOPK_EVAL:
        row[f'r@{k}']    = round(m[f'r{k}'] / cnt * 100, 4)
        row[f'ndcg@{k}'] = round(m[f'n{k}'] / cnt,       6)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)


# ── Build Per-Domain DataFrame ─────────────────────────────────────────────────

domain_rows = []
for domain in sorted(all_domain_metrics.keys()):
    dm = all_domain_metrics[domain]
    for key in STRATEGY_RATIO_KEYS:
        if key not in dm:
            continue
        m_   = dm[key]; cnt_ = m_['count'] or 1
        if key == 'baseline':
            strategy_name = 'baseline'; ratio_val = 1.0; pct_val = 100
        else:
            parts         = key.rsplit('_r', 1)
            strategy_name = parts[0]; pct_val = int(parts[1]); ratio_val = pct_val / 100.0
        row = {'domain': domain, 'method': key,
               'strategy': strategy_name, 'keep_ratio': ratio_val, 'keep_pct': pct_val,
               'n_queries': cnt_}
        for k in TOPK_EVAL:
            row[f'r@{k}']    = round(m_[f'r{k}'] / cnt_ * 100, 4)
            row[f'ndcg@{k}'] = round(m_[f'n{k}'] / cnt_,       6)
        domain_rows.append(row)

domain_df = pd.DataFrame(domain_rows)


# ── Build Per-Query DataFrame ──────────────────────────────────────────────────

query_df = pd.DataFrame(query_rows)


# ── Write CSV files ───────────────────────────────────────────────────────────

query_df_path   = os.path.join(WORKING_DIR, "compression_ratio_comparison_queries.csv")
summary_df_path = os.path.join(WORKING_DIR, "compression_ratio_comparison_summary.csv")
domain_df_path  = os.path.join(WORKING_DIR, "compression_ratio_comparison_domain.csv")

query_df.to_csv(query_df_path,   index=False)
summary_df.to_csv(summary_df_path, index=False)
domain_df.to_csv(domain_df_path,  index=False)

print(f"✅ Saved per-query results  : {query_df_path}")
print(f"✅ Saved overall summary    : {summary_df_path}")
print(f"✅ Saved per-domain summary : {domain_df_path}")


# ── Print final summary table ──────────────────────────────────────────────────

print_summary(all_metrics, STRATEGY_RATIO_KEYS,
              title="ViDoRe Compression Ratio Comparison — Full Summary")

print(f"""
Output files:
  {query_df_path}
  {summary_df_path}
  {domain_df_path}
""")

✅ Saved per-query results  : /kaggle/working/compression_ratio_comparison_queries.csv
✅ Saved overall summary    : /kaggle/working/compression_ratio_comparison_summary.csv
✅ Saved per-domain summary : /kaggle/working/compression_ratio_comparison_domain.csv

ViDoRe Compression Ratio Comparison — Full Summary
Method                                       R@1     R@3     R@5    R@10    nDCG@1    nDCG@3    nDCG@5   nDCG@10
--------------------------------------------------------------------------------------------------------
baseline                                  19.40%  33.50%  40.79%  50.88%   0.4262   0.4194   0.4277   0.4515
random_r10                                13.89%  24.76%  30.92%  39.73%   0.3278   0.3176   0.3254   0.3453
random_r20                                15.96%  28.38%  34.73%  44.39%   0.3674   0.3598   0.3656   0.3881
random_r30                                17.30%  29.77%  36.43%  46.10%   0.3867   0.3765   0.3847   0.4070
random_r40                           